# Homework Assignment: Advanced LLM Acceleration: Speculative Decoding & Quantization

**Target hardware:** 1x NVIDIA H100 80GB

**Base model:** `Qwen/Qwen3-8B`

## Objective

Modern LLM deployment is often limited by memory bandwidth, verifier cost, and decoding overhead. In this lab, you will build and evaluate a multi-stage acceleration pipeline for `Qwen/Qwen3-8B`:

- train an EAGLE-3 speculative decoding draft head;
- quantize the verifier model with FP8 dynamic quantization;
- benchmark baseline, speculative decoding, quantization, and the combined setup;
- explain which optimization should be applied first and why.

The main question for the assignment:

**Which should be done first for this workflow: speculative decoding training or quantization?**

Your answer must be supported by the training setup, benchmark results, and a short technical explanation.

---

## References

- Speculators library: <https://github.com/vllm-project/speculators>
- Offline EAGLE-3 training tutorial: <https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>
- FP8 dynamic quantization reference: <https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

---

## Required Library Versions

Use separate environments. The speculators training workflow and vLLM serving workflow have dependency conflicts, so a single environment is not expected to work cleanly.

| Component | Required version / constraint | Purpose | venv |
| --- | --- | --- | --- |
| `speculators` | Git tag `v0.5.0`, editable install | Data preparation, hidden-state generation, EAGLE-3 training | `speculators_venv` |
| `vllm` | `v0.20.0` | Serving and benchmark runtime | `vllm_venv` |
| `fastapi` | `<0.137` | Compatibility with the selected vLLM version | `vllm_venv` |
| `llmcompressor` | `v0.12.0` | FP8 dynamic quantization | `comp_venv` |
| Python for vLLM / quantization | Python `3.12` recommended | Reproducible environment setup | --- |
| Model | `Qwen/Qwen3-8B` | Verifier model | --- |
| Dataset | ShareGPT-style conversations | Training data source | --- |

Expected local environments:

- `speculators_venv`
- `vllm_venv`
- `comp_venv`

Do not submit the virtual environments.

Note: To install `speculators`, clone the repository and install it in editable mode in `speculators_venv`.

---

## Task 1: Environment & Data Orchestration

Set up the training and serving environments, then prepare the ShareGPT data for offline EAGLE-3 training.

Required work:

- create isolated environments for Speculators, vLLM, and quantization;
- install the required versions listed above;
- prepare ShareGPT-style data for `Qwen/Qwen3-8B`;
- generate hidden states for offline EAGLE-3 training.

Background:

Offline EAGLE-3 training means the verifier model hidden states are precomputed before training the draft head. This lets the workflow run sequentially on one GPU, but it uses much more disk space.

Online EAGLE-3 training computes hidden states during training. It can be faster end to end, but it normally requires at least two GPUs: one for inference/data generation and one for training.

Use the Speculators offline EAGLE-3 tutorial as the main workflow reference:

<https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline>

You need to implement an offline EAGLE-3 training pipeline.

Hints:

- Watch local disk usage. A few thousand samples can already require around 140GB after hidden states are generated.
- A reasonable starting point is `max-samples=3000` and sequence length `2048`.
- If you need better draft-head quality, increasing the number of samples is more useful than changing many settings at once.
- Use the `scripts/launch_vllm.py` script to run the vLLM server.

Question: Why do hidden states require much more disk space than the original text dataset?

Troubleshooting hints:

- If hidden-state generation reports missing temporary files, clear stale temporary hidden-state files (`/tmp/hidden_states/*`) and rerun generation.
- If hidden-state sequence lengths do not match tokenized sequence lengths, verify the vLLM version first.
- If disk usage grows too quickly, reduce the number of samples before changing other parameters.

---

# Task 2: Train an EAGLE-3 Draft Head

Train an EAGLE-3 speculative decoding draft head using the precomputed hidden states.

Required work:

- Use `Qwen/Qwen3-8B` as the verifier model for training.
- Save checkpoints under `output/checkpoints/`.
- Use the best checkpoint for serving and benchmarking.
- Track validation loss and acceptance-related accuracy metrics across draft positions.

Hints:

- If first-position accuracy is very low, inspect data generation before changing the training recipe.
- Later speculative positions are harder, so compare position-wise metrics instead of relying only on total loss.

Reference training result from one run:

| Metric | Value |
| --- | ---: |
| `val/loss_0_epoch` | `2.509` |
| `val/full_acc_0_epoch` | `0.463` |
| `val/cond_acc_0_epoch` | `0.463` |
| `val/loss_1_epoch` | `3.778` |
| `val/full_acc_1_epoch` | `0.181` |
| `val/cond_acc_1_epoch` | `0.364` |
| `val/loss_2_epoch` | `4.550` |
| `val/full_acc_2_epoch` | `0.069` |
| `val/cond_acc_2_epoch` | `0.320` |
| `val/loss_epoch` | `10.837` |
| Epoch | `4` |

Note: Epoch=4 means this is the 5th epoch, the index starts with 0.

Questions to answer:

1. What do `full_acc` and `cond_acc` measure in speculative decoding training?
2. Why does accuracy usually decrease for later speculative positions?
3. What would you change if the first-position accuracy is very low?

---

## Task 3: Quantize the Verifier Model

Quantize `Qwen/Qwen3-8B` using FP8 dynamic quantization.

Required work:

- Use `llmcompressor`.
- Apply FP8 dynamic quantization to linear layers.
- Keep `lm_head` unquantized.
- Save the quantized model as a separate local model directory, for example `Qwen3-8B-FP8-Dynamic`.
- Do not overwrite the original model checkpoint.

Reference material:

<https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md>

Hints:

- Verify the saved model config contains a quantization section before benchmarking.
- Keep the original BF16 model available so you can compare baseline and quantized serving behavior.

Expected quantization properties:

| Property | Expected value |
| --- | --- |
| Quantization method | compressed tensors |
| Weight format | FP8 |
| Activation format | dynamic FP8 |
| Target modules | linear layers |
| Ignored module | `lm_head` |

Questions to answer:

1. Why is FP8 dynamic quantization useful for serving on H100?
2. Why might `lm_head` be excluded from quantization?
3. How can quantization affect speculative decoding acceptance rate?

---

## Task 4: Serve and Benchmark

Benchmark four configurations:

1. Baseline `Qwen/Qwen3-8B`
2. `Qwen/Qwen3-8B` with the trained EAGLE-3 draft head
3. FP8 dynamically quantized `Qwen3-8B`
4. FP8 dynamically quantized `Qwen3-8B` with the trained EAGLE-3 draft head

Required work:

- Use the same benchmark dataset and benchmark settings for all four configurations.
- Keep concurrency fixed across experiments.
- Disable prefix caching unless you intentionally study its effect.
- Use a fixed seed where possible.
- Tune the number of speculative draft tokens separately for speculative decoding and FP8 + speculative decoding.

Important:

The reference results used different numbers of speculative draft tokens for the unquantized speculative-decoding run and the FP8 + speculative-decoding run. Do not assume one value is optimal for both. Tune it yourself and justify the final choice using throughput, acceptance rate, acceptance length, and TPOT.

Hints:

- Start with a small number of speculative tokens, then increase only if the acceptance behavior supports it.
- Compare output token throughput first, then use TTFT, TPOT, and acceptance metrics to explain the result.
- If a setting produces more draft work but little accepted work, it is probably not the best setting.

Script for benchmarking:

```bash
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --max-concurrency 8 \
    --dataset-path philschmid/mt-bench \
    --num-prompts 80
```


Reference benchmark results:

| Configuration | Duration, s | Requests/s | Output tok/s | Total tok/s | Mean TTFT, ms | Mean TPOT, ms | Acceptance rate |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Baseline | `24.35` | `3.29` | `841.22` | `1090.87` | `576.17` | `7.28` | N/A |
| Speculative decoding | `16.27` | `4.92` | `1258.65` | `1632.19` | `78.17` | `5.76` | `22.48%` |
| FP8 quantization | `13.06` | `6.12` | `1566.56` | `2031.82` | `51.18` | `4.90` | N/A |
| FP8 + speculative decoding | `11.59` | `6.90` | `1766.55` | `2290.82` | `30.24` | `4.28` | `36.50%` |

Reference speculative decoding details:

| Configuration | Reference draft tokens | Acceptance length | Drafts | Draft tokens | Accepted tokens |
| --- | ---: | ---: | ---: | ---: | ---: |
| Speculative decoding | `2` | `1.45` | `14088` | `28176` | `6334` |
| FP8 + speculative decoding | `1` | `1.36` | `14954` | `14954` | `5458` |

Your exact numbers may differ, but the relative comparison should be explainable.

Questions to answer:

1. Why can speculative decoding improve throughput even when acceptance rate is not close to 100%?
2. How many speculative tokens are optimal for this setup? Explain using acceptance rate, acceptance length, and TPOT.

---

## Final Report Requirements

Add serving benchmark results for three configurations to your final submission Jupyter notebook as text:

- speculative decoding;
- FP8 quantization;
- FP8 + speculative decoding.

---

## Scoring Rubric

Scores are based on `Output token throughput (tok/s)` from `vllm bench serve`.
Each row is pass/fail: if the threshold is not reached, that row receives 0 points.

| Requirement | Passing threshold | Points |
| --- | ---: | ---: |
| Speculative decoding result with trained EAGLE-3 draft head | `> 1250 tok/s` | 25 |
| FP8 dynamic quantization result | `> 1550 tok/s` | 10 |
| Best combined FP8 + speculative decoding result, with draft-token tuning and explanation | `> 1750 tok/s` | 15 |
| **Total** |  | **50** |




This is an example of the output we expect to be submitted. This is the result we get from the baseline model out of the box:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  24.35     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              3.29      
Output token throughput (tok/s):         841.22    
Peak output token throughput (tok/s):    1144.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1090.87   
---------------Time to First Token----------------
Mean TTFT (ms):                          576.17    
Median TTFT (ms):                        46.05     
P99 TTFT (ms):                           5677.04   
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          7.28      
Median TPOT (ms):                        7.01      
P99 TPOT (ms):                           11.66     
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.28      
Median ITL (ms):                         7.00      
P99 ITL (ms):                            7.68      
==================================================

```

Speculative decoding benchmark results:
```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  14.66     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              5.46      
Output token throughput (tok/s):         1397.10   
Peak output token throughput (tok/s):    1056.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1811.73   
---------------Time to First Token----------------
Mean TTFT (ms):                          60.02     
Median TTFT (ms):                        23.04     
P99 TTFT (ms):                           399.78    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.30      
Median TPOT (ms):                        5.35      
P99 TPOT (ms):                           6.03      
---------------Inter-token Latency----------------
Mean ITL (ms):                           7.66      
Median ITL (ms):                         7.62      
P99 ITL (ms):                            8.30      
---------------Speculative Decoding---------------
Acceptance rate (%):                     22.45     
Acceptance length:                       1.45      
Drafts:                                  14100     
Draft tokens:                            28200     
Accepted tokens:                         6331      
Per-position acceptance (%):
  Position 0:                            35.94     
  Position 1:                            8.96      
==================================================
```

FP8 quantization benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  11.99     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              6.67      
Output token throughput (tok/s):         1707.76   
Peak output token throughput (tok/s):    1752.00   
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          2214.58   
---------------Time to First Token----------------
Mean TTFT (ms):                          27.14     
Median TTFT (ms):                        23.52     
P99 TTFT (ms):                           54.64     
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          4.59      
Median TPOT (ms):                        4.59      
P99 TPOT (ms):                           4.66      
---------------Inter-token Latency----------------
Mean ITL (ms):                           4.59      
Median ITL (ms):                         4.58      
P99 ITL (ms):                            5.02      
==================================================
```

FP8 + speculative decoding benchmark results:

```
============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  10.60     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              7.54      
Output token throughput (tok/s):         1931.37   
Peak output token throughput (tok/s):    1528.00   
Peak concurrent requests:                17.00     
Total token throughput (tok/s):          2504.55   
---------------Time to First Token----------------
Mean TTFT (ms):                          45.22     
Median TTFT (ms):                        21.03     
P99 TTFT (ms):                           263.19    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          3.83      
Median TPOT (ms):                        3.85      
P99 TPOT (ms):                           4.40      
---------------Inter-token Latency----------------
Mean ITL (ms):                           5.20      
Median ITL (ms):                         5.15      
P99 ITL (ms):                            7.32      
---------------Speculative Decoding---------------
Acceptance rate (%):                     35.73     
Acceptance length:                       1.36      
Drafts:                                  15048     
Draft tokens:                            15048     
Accepted tokens:                         5376      
Per-position acceptance (%):
  Position 0:                            35.73     
==================================================
```

# Main Question:
`Which should be done first: speculative-decoding training or quantization?`


I think it makes most sense to quantize first. The two steps aren't the same in terms of cost — FP8 dynamic quantization is basically free (a one pass / few minutes, no calibration data etc), whereas training the draft head is the expensive part. Generating around 120 GB of hidden states plus several epochs of training all cost time / effort. There's also an alignment argument which is potentially stronger - the EAGLE-3 head is trained to mimic one specific verifier, and acceptance judged by that verifier's output distribution, so you would prefer to train against the exact model you'll deploy. If you quantize first, the hidden states you cache and the head's training target both come from the FP8 model you actually serve — no train/serve mismatch, and no risk of having to re-regenerate the 120 GB later. In our case the BF16-trained head transferred to FP8 with almost no drop in acceptance, so the mismatch penalty turned out to be small but the cost asymmetry alone makes quantize-first the sensible order.

# Task 1
`Why do hidden states need more disk than text dataset?`
I think this is just down to the amount of data in each representation. Each token is represented as an integer (token id) of say 2 bytes while the multiple hidden layers of Qwen's activations (4096 elements in BF16) required by Eagle-3 is approx. 3 * 4096 * 2 = 24 kB per token. This is roughly 6000 times the size.

# Task 2
`full_acc` - accuracy at a token position cumulatively i.e. p1 * p2 * ... pn

`cond_acc` - this measures how often the draft head's pick for the next token is deemed correct by the verifier, given the previous tokens were correct. i.e. the per-step accuracy.

`Why does accuracy fall for later positions?`
Predicting further into the future is naturally more difficult but also because errors that the head makes at earlier positions propogate along the prediction chain. This affects 'full_acc' most since it is a product of each probability.

`What if first-position accuracy is very low?`
If the position 1 accuracy is very low it's probably because the hidden states are misaligned with the tokens i.e. there's a systematic problem in the way the dataset was built. We would expect the head to have some success in predicting the first tokens from the models hidden states.

# Task 3
`Why is FP8 dynamic quantization useful on H100?`
H100 has native FP8 tensor cores, so FP8 matmuls run at roughly twice BF16 throughput. Also decoding is memory bandwidth bound and FP8 halves bytes versus BF16 so an obvious win since it ~halves weight traffic from HBM per token.

`Why exclude lm_head from quantization?`
A reduction in precision here would adversely affect the choice of next token and degrade the performance. This would happen because the head maps the hidden state onto the logits so a drop in precision would affect the output distribution / which tokens are emitted / accepted.

`How can quantization affect speculative acceptance rate?`
Quantizing the verifier shifts that distribution slightly from the BF16 model the head was trained against which means the draft's guesses match a bit less often and so acceptance falls. In our runs though the effect was negligible - at matched draft count FP8 and BF16 had essentially the same acceptance length (~1.36 at 1 draft token, ~1.45 at 2), so the effect was small in our setup. 

# Task 4
`Why can speculative decoding help even when acceptance is well below 100%?`
As long as the acceptance length > 1 we are getting some positive value out of the speculative decoding setup. One verifier fwd pass checks all speculative tokens in parallel and since the draft model is really cheap by comparison, being able to do this with only one HBM pass processes a lot of tokens at low cost. If we manage to emit more than one token per expensive verifier pass and amortize its HBM, we get higher throughput with a low breakeven hence the benefit at low acceptance rates.

`How many speculative tokens are optimal, and why?`
I tuned the number of speculative tokens separately for the unquantized and FP8 runs by benchmarking each at different draft-token counts and comparing output throughput, acceptance length, and TPOT. For the FP8 + speculative configuration the optimum was clearly 1 draft token: it reached 1931 tok/s, versus 1819 tok/s at 2 draft tokens. For the unquantized speculative run, 1 and 2 draft tokens were effectively tied (both with 1250-1280 tok/s, within noise) and 3 was slightly slower (1244), so I used 2.

The reason is that each additional draft position is accepted far less often than the one before it. The per-position acceptance figures show this directly: position 0 is accepted 36% of the time, position 1 only 9%, and position 2 1% which can be seen in the section of `spec_ntok3.txt` below. As a result, drafting more tokens barely raises the acceptance length (1.36 with 1 token, 1.45 with 2, 1.47 with 3), while it increases the draft and verification work per step. Past a point as so little is being accepted, the extra overhead isn't repaid and throughput falls. This effect is stronger under FP8, because FP8 makes the verifier step so cheap that the cost of a second draft position outweighs its tiny acceptance gain, which is exactly why the FP8 run prefers fewer draft tokens (1) than the unquantized run (2).

```

  ============ Serving Benchmark Result ============
Successful requests:                     80        
Failed requests:                         0         
Maximum request concurrency:             8         
Benchmark duration (s):                  16.46     
Total input tokens:                      6078      
Total generated tokens:                  20480     
Request throughput (req/s):              4.86      
Output token throughput (tok/s):         1244.47   
Peak output token throughput (tok/s):    928.00    
Peak concurrent requests:                16.00     
Total token throughput (tok/s):          1613.80   
---------------Time to First Token----------------
Mean TTFT (ms):                          60.86     
Median TTFT (ms):                        26.07     
P99 TTFT (ms):                           370.21    
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          5.94      
Median TPOT (ms):                        6.00      
P99 TPOT (ms):                           6.98      
---------------Inter-token Latency----------------
Mean ITL (ms):                           8.70      
Median ITL (ms):                         8.64      
P99 ITL (ms):                            9.53      
---------------Speculative Decoding---------------
Acceptance rate (%):                     15.52     
Acceptance length:                       1.47      
Drafts:                                  13940     
Draft tokens:                            41820     
Accepted tokens:                         6491      
Per-position acceptance (%):
  Position 0:                            36.31     
  Position 1:                            8.70      
  Position 2:                            1.55      
==================================================

```